# Plant memory demo

This adds durable plant and supply memory; previously the repository had only a spec. Events and current state share a transaction so a retry cannot leave conflicting facts. Corrections preserve the original event, and approximate quantities remain descriptive. Care advice stays with the LLM because the server's job is to remember reported facts.

Input comes from `data/demo_garden.json`, using the spec's examples. No actual private plant data was provided, so this fixture is explicitly fictional. The demo uses an isolated schema in TEST_DATABASE_URL and removes that schema when finished.

In [1]:
import json
import os
from datetime import UTC, datetime, timedelta
from pathlib import Path
from uuid import UUID, uuid4

import psycopg
from psycopg import sql
from psycopg.conninfo import make_conninfo

from her_garden.models import InventoryEvent, PlantEvent, PlantState
from her_garden.store import GardenStore

root = Path.cwd()
fixture = json.loads((root / "data/demo_garden.json").read_text())
print(fixture["provenance"])
print("Plant:", fixture["plant"]["name"], "| Location:", fixture["location"])

Public demonstration fixture based on examples in docs/plant_memory_mcp_spec.md; not private household data.
Plant: Crassula | Location: north balcony


In [2]:
async def show_demo() -> None:
    dsn = os.environ["TEST_DATABASE_URL"]
    schema = "demo_" + uuid4().hex
    async with await psycopg.AsyncConnection.connect(dsn, autocommit=True) as conn:
        await conn.execute(sql.SQL("CREATE SCHEMA {}").format(sql.Identifier(schema)))
    store = GardenStore(make_conninfo(dsn, options=f"-csearch_path={schema}"))
    await store.open()
    try:
        location = await store.create_location(uuid4(), fixture["location"])
        plant = await store.create_plant(
            uuid4(), PlantState(**fixture["plant"], location_id=UUID(location["entity_id"]))
        )
        plant_id = UUID(plant["entity_id"])
        now = datetime.now(UTC)
        request_id = uuid4()
        event = PlantEvent(
            event_type="watering", occurred_at=now - timedelta(days=2), note="Watered thoroughly"
        )
        saved = await store.append_plant_event(request_id, plant_id, event)
        retry = await store.append_plant_event(request_id, plant_id, event)
        print("Technical retry returns the same event:", saved == retry)
        await store.append_plant_event(
            uuid4(),
            plant_id,
            PlantEvent(
                event_type="watering",
                occurred_at=now - timedelta(days=9),
                note="Earlier watering reported later",
            ),
        )
        context = await store.get_plant_context(plant_id)
        print(
            "Backdated report preserves latest watering:",
            context["latest_actions"]["watering"]["event_id"] == saved["event_id"],
        )
        await store.append_plant_event(
            uuid4(),
            plant_id,
            PlantEvent(
                event_type="void",
                occurred_at=now,
                note="Actually watered another plant",
                supersedes_event_id=UUID(saved["event_id"]),
            ),
        )
        context = await store.get_plant_context(plant_id)
        print("After correction:", context["latest_actions"]["watering"]["details"]["note"])
        print("Original history retained:", context["total_events"], "events")
        item = await store.append_inventory_event(
            uuid4(),
            None,
            InventoryEvent(
                **fixture["inventory"],
                event_type="observation",
                occurred_at=now,
                note="Checked cupboard",
            ),
        )
        print("Perlite remaining:", (await store.list_entities("inventory"))[0]["remaining"])
        await store.append_inventory_event(
            uuid4(),
            UUID(item["entity_id"]),
            InventoryEvent(event_type="usage", occurred_at=now, note="Used some for repotting"),
        )
        print(
            "After unspecified usage:",
            (await store.list_entities("inventory"))[0]["remaining"],
            "(unknown, not zero)",
        )
    finally:
        await store.close()
        async with await psycopg.AsyncConnection.connect(dsn, autocommit=True) as conn:
            await conn.execute(sql.SQL("DROP SCHEMA {} CASCADE").format(sql.Identifier(schema)))


await show_demo()

Technical retry returns the same event: True


Backdated report preserves latest watering: True


After correction: Earlier watering reported later
Original history retained: 4 events


Perlite remaining: about half a bag


After unspecified usage: None (unknown, not zero)
